# IFS cut-off activity from DAILY-MEAN Z500 (vs. 6-hourly)

Recomputes the colindex2 cut-off-activity index by running the detector on the
**daily-mean** Z500 fields (one field per day, 20-25 May) instead of the native
6-hourly fields, then correlates the resulting COL activity with the 25 May
target-box T2m -- raw and ridge-controlled (residual | Z500) -- for every window
`COL_d0..COL_d5`. This matches the daily-mean detection already used for the
binary cut-off classification and for the flow analogues.

Runs on the cluster (needs `colindex2` and the TIGGE Z500 GRIB). Detector
settings, region box and thresholds are identical to the 6-hourly notebook -- the
**only** change is that each field fed to the detector is a 24 h mean.

In [ ]:
import os
from pathlib import Path
from contextlib import contextmanager
import numpy as np, pandas as pd, xarray as xr
from scipy import stats
from joblib import Parallel, delayed
os.environ["PATH"] = "/path/to/cdo/bin" + os.pathsep + os.environ.get("PATH", "")

# ---- paths ----
S2S_DIR   = "${IFS_FORECAST_DIR}"
# per-day files: z500 on 1x1, t2m on 0.5x0.5 (one file per init day)
Z500_TMPL = os.path.join(S2S_DIR, "tigge_ecmwf_pf_gh500_6h_{date}_1x1.grib")
T2M_TMPL  = os.path.join(S2S_DIR, "tigge_ecmwf_pf_2m_temperature_sfc_6h_{date}_05x05.grib")
WORK      = os.path.join(S2S_DIR, "ifs_cutoff_activity"); os.makedirs(WORK, exist_ok=True)
member_dim, init_dim, lead_dim = "number", "time", "step"; g0 = 9.80665

# ---- event / window ----
INITS = [pd.Timestamp("2026-05-17"), pd.Timestamp("2026-05-18")]
DAYS  = pd.date_range("2026-05-20", "2026-05-25", freq="D")   # 6 daily-mean fields
END   = pd.Timestamp("2026-05-25")

# ---- colindex2 detector: EXACTLY the 6-hourly / analogue settings ----
COL_INPUT_IS_GEOPOTENTIAL = False        # TIGGE 'gh' is geopotential HEIGHT in metres
COL_LON_MIN, COL_LON_MAX = -35, 0        # classification box (features counted here)
COL_LAT_MIN, COL_LAT_MAX = 25, 50
COL_DETECTION_LON_MIN, COL_DETECTION_LON_MAX = -50, 55
COL_DETECTION_LAT_MIN, COL_DETECTION_LAT_MAX = 5, 90
COL_RADII_KM = np.arange(200, 2101, 100)
# was:
#COL_DETECTION_SO_MIN, COL_CUTOFF_SO_MIN, COL_TROUGH_SO_MIN, COL_SR_MAX = 3.0, 6.0, 6.0, 3.0
# ex==1 only:
COL_DETECTION_SO_MIN, COL_CUTOFF_SO_MIN, COL_TROUGH_SO_MIN, COL_SR_MAX = 3.0, 6.0, np.inf, 3.0
COL_REMOVE_MIN_RADIUS = COL_REMOVE_MAX_RADIUS = True
COL_USE_GPU = False; COL_CHUNK_SIZE = 100_000
MEMBER_LIMIT = None; N_WORKERS = 16
print("daily-mean detection | days:", [f"{d:%m-%d}" for d in DAYS])

In [ ]:
from colindex2 import Detect

def select_latitude_range(field, lo, hi):
    return field.sel(lat=slice(lo, hi) if bool(field["lat"][0] < field["lat"][-1]) else slice(hi, lo))

@contextmanager
def colindex_working_directory(path):
    prev = Path.cwd(); path = Path(path); path.mkdir(parents=True, exist_ok=True); os.chdir(path)
    try: yield
    finally: os.chdir(prev)

def run_z500_detector(height, output_directory, stencil_directory):
    """colindex2 detector -- identical settings to the 6-hourly notebook."""
    if "time" in height.dims and height.sizes["time"] == 1:
        height = height.isel(time=0)
    with colindex_working_directory(stencil_directory):
        Detect(height, odir=str(output_directory), ty="L", r=COL_RADII_KM,
               SR_thres=COL_SR_MAX, So_thres=COL_DETECTION_SO_MIN, Do_thres=0.0, xx_thres=0.0,
               rm_rmin=COL_REMOVE_MIN_RADIUS, rm_rmax=COL_REMOVE_MAX_RADIUS,
               local_ex_req_num=7, gpu=COL_USE_GPU, chunk_size=COL_CHUNK_SIZE, nc=True, subs=False)

def colindex_csv_path(output_directory, timestamp):
    ts = pd.Timestamp(timestamp)
    return (Path(output_directory) / "V" / ts.strftime("%Y%m") / ("V-L-" + ts.strftime("%Y%m%d%H%M") + "-0500.csv"))

def read_field_So(output_directory, timestamp):
    """Per-field: (sum_So over qualifying dynamic lows in box, sum_So cut-offs only, max_So, n_features)."""
    csv = colindex_csv_path(output_directory, timestamp)
    if not csv.exists(): return 0.0, 0.0, 0.0, 0
    pts = pd.read_csv(csv)
    if pts.empty: return 0.0, 0.0, 0.0, 0
    lon = (pts["lon"] + 180) % 360 - 180
    in_box = lon.between(COL_LON_MIN, COL_LON_MAX) & pts["lat"].between(COL_LAT_MIN, COL_LAT_MAX)
    ex = pd.to_numeric(pts["ex"], errors="coerce")
    applied = np.where(ex.eq(1), COL_CUTOFF_SO_MIN, COL_TROUGH_SO_MIN)
    qual = pts["So"].ge(applied) & in_box
    so = pts.loc[qual, "So"]; so_cut = pts.loc[qual & ex.eq(1), "So"]
    return (float(so.sum()), float(so_cut.sum()), float(so.max()) if len(so) else 0.0, int(len(so)))

In [ ]:
# ---- open the IFS Z500 ensemble (per-init 1x1 files), stack along init 'time' ----
def _open_z500_init(it):
    dsi = xr.open_dataset(Z500_TMPL.format(date=f"{it:%Y%m%d}"), engine="cfgrib", backend_kwargs={"indexpath": ""})
    v = dsi["gh"] if "gh" in dsi else (dsi["z"] if "z" in dsi else list(dsi.data_vars.values())[0])
    if "time" not in v.dims: v = v.expand_dims("time")
    return v.drop_vars("valid_time", errors="ignore")     # daymean_field recomputes valid_time = init + step
z500 = xr.concat([_open_z500_init(it) for it in INITS], dim="time")
ren = {}
if "longitude" in z500.coords: ren["longitude"] = "lon"
if "latitude"  in z500.coords: ren["latitude"]  = "lat"
z500 = z500.rename(ren)
z500 = z500.assign_coords(lon=((z500["lon"] + 180) % 360) - 180).sortby("lon").sortby("lat")
MEMBERS = [int(m) for m in np.atleast_1d(z500[member_dim].values)]
if MEMBER_LIMIT: MEMBERS = MEMBERS[:MEMBER_LIMIT]
print("variable:", z500.name, "| units:", z500.attrs.get("units"), "| members:", len(MEMBERS),
      "| grid:", round(float(abs(z500['lat'][1]-z500['lat'][0])),3), "deg")

In [ ]:
# ---- DAILY-MEAN detection: one 24 h-mean field per day, over the window ----
DAYMEAN_STENCIL = Path(WORK) / "stencil_daymean_window"

def daymean_field(init_ts, mem, date):
    """24 h mean (00/06/12/18) of Z500 for one (init, member, day), cropped to the detection box."""
    sub = z500.sel({init_dim: np.datetime64(init_ts), member_dim: mem})
    vt = (pd.DatetimeIndex(np.atleast_1d(sub["valid_time"].values)) if "valid_time" in sub.coords
          else pd.DatetimeIndex(init_ts + pd.to_timedelta(np.atleast_1d(sub[lead_dim].values))))
    day = pd.Timestamp(date).normalize(); idx = np.where(vt.normalize() == day)[0]
    if len(idx) == 0: return None
    fld = sub.isel({lead_dim: idx}).mean(lead_dim)                                   # <-- the daily mean
    fld = fld.drop_vars([c for c in (init_dim, member_dim, "valid_time") if c in fld.coords], errors="ignore")
    fld = fld.expand_dims(time=[day]).sel(lon=slice(COL_DETECTION_LON_MIN, COL_DETECTION_LON_MAX))
    fld = select_latitude_range(fld, COL_DETECTION_LAT_MIN, COL_DETECTION_LAT_MAX)
    if COL_INPUT_IS_GEOPOTENTIAL: fld = fld / g0
    for d in list(fld.dims):
        if fld.sizes[d] == 1 and d not in ("lat", "lon", "time"): fld = fld.squeeze(d, drop=True)
    return fld.assign_attrs(units="m").assign_coords(level=500).transpose("time", "lat", "lon")

def daily_window_stack(init_ts, mem, days=DAYS):
    flds = [f for f in (daymean_field(init_ts, mem, D) for D in days) if f is not None]
    return xr.concat(flds, dim="time") if flds else None

def process_member_daily(init_ts, mem):
    odir = Path(WORK) / "det_daymean_window" / f"init{init_ts:%Y%m%d}_m{int(mem):02d}"
    stack = daily_window_stack(init_ts, mem)
    if stack is None: return None
    run_z500_detector(stack.load(), odir, DAYMEAN_STENCIL)
    per = pd.DataFrame([(D,) + read_field_So(odir, D) for D in DAYS],
                       columns=["day", "sum_So", "sum_So_cutoff", "max_So", "n_feat"])
    row = {"init": f"{init_ts:%Y-%m-%d}", "member": int(mem)}
    for k in range(6):                                       # COL_dk = window [25-k, 25], now in DAILY fields
        row[f"COL_d{k}"] = float(per.loc[per["day"] >= END - pd.Timedelta(days=k), "sum_So"].sum())
    row["cutoff25"] = bool(per.loc[per["day"] == END, "n_feat"].iloc[0] > 0)   # daily-mean presence on the 25th
    return row

# clear any stale stencil cache (e.g. from a different grid) so colindex2 regenerates it
import shutil
if DAYMEAN_STENCIL.exists(): shutil.rmtree(DAYMEAN_STENCIL)
DAYMEAN_STENCIL.mkdir(parents=True, exist_ok=True)
# warm the stencil once, then run all members in parallel
_f0 = daily_window_stack(INITS[0], MEMBERS[0])
run_z500_detector(_f0.isel(time=[0]), Path(WORK) / "det_daymean_window" / "_warmup", DAYMEAN_STENCIL)
recs = Parallel(n_jobs=N_WORKERS, verbose=5)(delayed(process_member_daily)(it, m) for it in INITS for m in MEMBERS)
daily_act = pd.DataFrame([r for r in recs if r]).sort_values(["init", "member"]).reset_index(drop=True)
daily_act.to_csv(os.path.join(WORK, "ifs_COL_window_sweep_DAILYMEAN.csv"), index=False)
print("saved ifs_COL_window_sweep_DAILYMEAN.csv | members:", len(daily_act),
      "| cutoff25 present:", int(daily_act.cutoff25.sum()))

In [ ]:
# ---- 25 May target-box daily-mean T2m [C] and Z500 index [gpm] (detection-independent) ----
box = (-5.5, 5, 44.5, 55); VALID = pd.Timestamp("2026-05-25")

def target_box_daymean(tmpl, varnames, lon0, lon1, lat0, lat1, to_celsius=False):
    rows = []
    for it in INITS:
        d = xr.open_dataset(tmpl.format(date=f"{it:%Y%m%d}"), engine="cfgrib", backend_kwargs={"indexpath": ""})
        v = next((d[n] for n in varnames if n in d), list(d.data_vars.values())[0])
        ren = {}
        if "longitude" in v.coords: ren["longitude"] = "lon"
        if "latitude"  in v.coords: ren["latitude"]  = "lat"
        v = v.rename(ren).assign_coords(lon=lambda x: ((x["lon"] + 180) % 360) - 180).sortby("lon").sortby("lat")
        vt = (pd.DatetimeIndex(np.atleast_1d(v["valid_time"].values)) if "valid_time" in v.coords
              else pd.DatetimeIndex(it + pd.to_timedelta(np.atleast_1d(v[lead_dim].values))))
        idx = np.where(vt.floor("D") == VALID)[0]
        day = v.isel({lead_dim: idx}).mean(lead_dim)
        reg = day.sel(lon=slice(lon0, lon1), lat=select_latitude_range(day, lat0, lat1)["lat"])
        w = np.cos(np.deg2rad(reg["lat"])); bm = reg.weighted(w).mean(("lat", "lon"))
        for mem in MEMBERS:
            rows.append({"init": f"{it:%Y-%m-%d}", "member": int(mem), "val": float(bm.sel({member_dim: mem}).values)})
    df = pd.DataFrame(rows)
    if to_celsius: df["val"] = df["val"] - 273.15
    return df

t2m = target_box_daymean(T2M_TMPL, ["t2m", "2t"], *box, to_celsius=True).rename(columns={"val": "t2m_C"})
z5  = target_box_daymean(Z500_TMPL, ["gh", "z"], *box).rename(columns={"val": "z500_gh"})
m = daily_act.merge(t2m, on=["init", "member"]).merge(z5, on=["init", "member"])
print("merged:", len(m))

In [ ]:
# ---- correlations: DAILY-MEAN COL activity vs 25 May T2m (raw & ridge-removed) ----
def resid(y, x):
    A = np.column_stack([np.ones(len(y)), x]); return y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
T = m["t2m_C"].values; Z = m["z500_gh"].values; Tr = resid(T, Z); n = len(m)
print(f"DAILY-MEAN detection, n={n}")
print(f"Z500 vs raw T2m : Pearson {stats.pearsonr(Z,T)[0]:+.3f} | Spearman {stats.spearmanr(Z,T)[0]:+.3f}\n")
print(f"{'win':4} {'raw r':>8} {'raw rho':>8} | {'|Z r':>8} {'|Z rho':>8}")
for k in range(6):
    C = m[f"COL_d{k}"].values
    print(f"d{k:<3} {stats.pearsonr(C,T)[0]:+8.3f} {stats.spearmanr(C,T)[0]:+8.3f} | "
          f"{stats.pearsonr(C,Tr)[0]:+8.3f} {stats.spearmanr(C,Tr)[0]:+8.3f}")

# ---- side-by-side with the 6-HOURLY detection, if that sweep CSV exists ----
p6 = os.path.join(WORK, "ifs_COL_window_sweep.csv")
if os.path.exists(p6):
    m6 = pd.read_csv(p6).merge(m[["init", "member", "t2m_C", "z500_gh"]], on=["init", "member"])
    T6 = m6["t2m_C"].values; Z6 = m6["z500_gh"].values; Tr6 = resid(T6, Z6)
    print("\n6-HOURLY vs DAILY-MEAN (Pearson r):")
    print(f"{'win':4} {'raw 6h':>8} {'raw day':>8} | {'|Z 6h':>8} {'|Z day':>8}")
    for k in range(6):
        r6r = stats.pearsonr(m6[f'COL_d{k}'].values, T6)[0]; rdr = stats.pearsonr(m[f'COL_d{k}'].values, T)[0]
        r6z = stats.pearsonr(m6[f'COL_d{k}'].values, Tr6)[0]; rdz = stats.pearsonr(m[f'COL_d{k}'].values, Tr)[0]
        print(f"d{k:<3} {r6r:+8.3f} {rdr:+8.3f} | {r6z:+8.3f} {rdz:+8.3f}")
else:
    print("\n[note] 6-hourly sweep CSV not found -> run the 6-hourly notebook first for the comparison")